In [1]:
import sys 
from pathlib import Path 

project_root = Path.cwd().parent 
sys.path.insert(0, str(project_root))

from src.evaluate_models import eval_reg 

import numpy as np 
import pandas as pd 

from sklearn.model_selection import train_test_split
from sklearn.ensemble import (
    ExtraTreesRegressor,
    GradientBoostingRegressor
)

print("Required libraries imported successfully")

Required libraries imported successfully


In [2]:
df = pd.read_csv(
    "../data/student-mat.csv",
    sep=";"
)

df_encoded = pd.get_dummies(
    df,
    drop_first=True
)

X_full = df_encoded.drop(
    columns=["G3"]
).copy()

y = df_encoded["G3"].copy()

print("Dataset loaded.")
print("X_full shape:", X_full.shape)

Dataset loaded.
X_full shape: (395, 41)


In [3]:
Xtr_f, Xte_f, ytr, yte = train_test_split(
    X_full,
    y,
    test_size=0.20,
    random_state=42
)

print("Training feature shape:", Xtr_f.shape)
print("Test feature shape:", Xte_f.shape)
print("Training target shape:", np.asarray(ytr).shape)
print("Test target shape:", np.asarray(yte).shape)

Training feature shape: (316, 41)
Test feature shape: (79, 41)
Training target shape: (316,)
Test target shape: (79,)


In [4]:
required_variables = [
    "Xtr_f",
    "Xte_f",
    "ytr",
    "yte"
]

missing_variables = [
    variable
    for variable in required_variables
    if variable not in globals()
]

if missing_variables:
    raise NameError(
        "Missing variables: "
        + ", ".join(missing_variables)
    )

print("All required modeling variables are available.")

All required modeling variables are available.


In [5]:
extra_trees_model = ExtraTreesRegressor(
    n_estimators=300,
    random_state=42
)

extra_trees_model.fit(
    Xtr_f,
    ytr
)

print("Extra Trees model trained successfully.")

Extra Trees model trained successfully.


In [6]:
extra_trees_predictions = (
    extra_trees_model.predict(Xte_f)
)

print(
    "Number of predictions:",
    len(extra_trees_predictions)
)

print(
    "First five predictions:"
)

print(
    extra_trees_predictions[:5]
)

Number of predictions: 79
First five predictions:
[ 7.68333333 12.01        5.10333333  9.51666667  7.52666667]


In [7]:
extra_trees_metrics = eval_reg(
    yte, 
    extra_trees_predictions
)

print("Extra Trees results")
print(extra_trees_metrics)

Extra Trees results
{'MAE': 1.3311814345991562, 'RMSE': 2.2644825354383293, 'R2': 0.7499210274296113}


In [8]:
gradient_boosting_model = GradientBoostingRegressor(
    n_estimators=300,
    random_state=42
)

gradient_boosting_model.fit(
    Xtr_f,
    ytr
)

print("Gradient Boosting model trained successfully.")

Gradient Boosting model trained successfully.


In [9]:
gradient_boosting_predictions = (
    gradient_boosting_model.predict(Xte_f)
)

print(
    "Number of predictions:",
    len(gradient_boosting_predictions)
)

print(
    "First five predictions:"
)

print(
    gradient_boosting_predictions[:5]
)

Number of predictions: 79
First five predictions:
[ 9.42858264 11.29650256  5.19400171 10.35330218  9.38024534]


In [10]:
gradient_boosting_metrics = eval_reg(
    yte, 
    gradient_boosting_predictions
)

print("Gradient Boosting results")
print(gradient_boosting_metrics)

Gradient Boosting results
{'MAE': 1.267622292850229, 'RMSE': 2.154159988807915, 'R2': 0.7736944862054645}


In [11]:
session29_results = pd.DataFrame([
    {
        "Model": "Extra Trees",
        "MAE": extra_trees_metrics["MAE"],
        "RMSE": extra_trees_metrics["RMSE"],
        "R2": extra_trees_metrics["R2"]
    },
    {
        "Model": "Gradient Boosting",
        "MAE": gradient_boosting_metrics["MAE"],
        "RMSE": gradient_boosting_metrics["RMSE"],
        "R2": gradient_boosting_metrics["R2"]
    }
])

session29_results.round(4)

,Model,MAE,RMSE,R2
0,Extra Trees,1.3312,2.2645,0.7499
1,Gradient Boosting,1.2676,2.1542,0.7737


In [12]:
all_models = pd.DataFrame([
    {
        "Model": "KNN",
        "RMSE": 3.3926950565642415
    },
    {
        "Model": "SVR",
        "RMSE": 2.7260297218073055
    },
    {
        "Model": "Decision Tree",
        "RMSE": 2.467248497769575
    },
    {
        "Model": "Random Forest",
        "RMSE": 1.97473322451841
    },
    {
        "Model": "Extra Trees",
        "RMSE": extra_trees_metrics["RMSE"]
    },
    {
        "Model": "Gradient Boosting",
        "RMSE": gradient_boosting_metrics["RMSE"]
    }
])

all_models = (
    all_models
    .sort_values("RMSE")
    .reset_index(drop=True)
)

all_models["Rank"] = range(
    1,
    len(all_models) + 1
)

all_models

,Model,RMSE,Rank
0,Random Forest,1.974733,1
1,Gradient Boosting,2.154160,2
2,Extra Trees,2.264483,3
3,Decision Tree,2.467248,4
4,SVR,2.726030,5
5,KNN,3.392695,6
